# Class 5: More pandas, data visualization, and interactive graphics

Plan for today is to:

1. Learn a few additional panadas operations, ways to visualize data
2. If there is time: discuss interactive graphics. 

As always, please run the cells below to download the data we will use today, and to load the package we will use. 


In [6]:
import YData_baseball

YData_baseball.download_data("Lahman_2024u/Batting.csv")
YData_baseball.download_data("Lahman_2024u/People.csv")

# A helper function to download the retrosheet play-by-play data
def download_retro_data(year):
    
    import os.path
    retro_file_name = str(year) + "plays.zip"
    if not os.path.isfile(retro_file_name):
        import requests
        retro_url = "https://www.retrosheet.org/downloads/plays/" + retro_file_name
        r = requests.get(retro_url )
        with open(retro_file_name, "wb") as f:
            f.write(r.content)
    else:
        print("File already downloaded, skipping download step.")


download_retro_data(2019)


The file `Batting.csv` already exists.
If you would like to download a new copy of the file, please rename the existing copy of the file.
The file `People.csv` already exists.
If you would like to download a new copy of the file, please rename the existing copy of the file.
File already downloaded, skipping download step.


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Part 1: Pivot tables and heatmaps

To start, let's explore creating pivot tables and visualizing these pivot tables as heatmaps. To do this, we will use the retrosheet play-by-play data from the 2019 MLB season which is loaded below. 


In [8]:
# Using dtype to avoid a warning about the umplf and umprf columns

retro_data = pd.read_csv("2019plays.zip", dtype={'umplf': "string", 'umprf': "string"})  

retro_data.head()


,gid,event,inning,top_bot,vis_home,site,batteam,pitteam,score_v,score_h,...,pn,umphome,ump1b,ump2b,ump3b,umplf,umprf,date,gametype,pbp
0,OAK201903200,9/L9M+,1,0,0,TOK01,SEA,OAK,0,0,...,1,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
1,OAK201903200,3/F3D,1,0,0,TOK01,SEA,OAK,0,0,...,2,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
2,OAK201903200,S6/L4D+,1,0,0,TOK01,SEA,OAK,0,0,...,3,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
3,OAK201903200,WP.1-2,1,0,0,TOK01,SEA,OAK,0,0,...,4,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
4,OAK201903200,K,1,0,0,TOK01,SEA,OAK,0,0,...,5,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full


## 1.1 Calculate the batting average separately for each ball-strike count

Let's write code to calculate the batting average for each ball-strike count. In particular, we will create a DataFrame where the index is the number of strikes, the columns are the number of balls, and the values are the batting average for that ball-strike count. 

The batting average will be calculated from all players who had at-bats in the 2019 season. Recall batting average is defined as: AVG = H/AB 

Here are the steps we will use to calculate the batting average for each ball-strike count:

1. Create a DataFrame `retro_at_bats` that only contains at-bats (i.e., `ab` is 1) 
2. Because there is some spurious data, reduce the `retro_at_bats` to only have rows where the number of balls is less than 4 (i.e., `balls < 4`)
3. Add a column to the `retro_at_bats` called `hits` which is the sum of singles, doubles, triples, and home runs
4. Reduce `retro_at_bats` to only the columns `balls`, `strikes`, and `hits`
5. Create a pivot table `ba_df` that has the index as `strikes`, the columns as `balls`, and the values as the mean of `hits`


In [9]:

# 1. Create the `retro_at_bats` DataFrame that only contains at-bats



# 2. Reduce the `retro_at_bats` to only have rows where the number of balls is less than 4



# 3. Add a column to the `retro_at_bats` called `hits` which is the sum of singles, doubles, triples, and home runs



# 4. Reduce `retro_at_bats` to only the columns `balls`, `strikes`, and `hits`



# 5. Create a pivot table `ba_df` that has the index as `strikes`, the columns as `balls`, and the values as the mean of `hits`





## 1.2 Visualize batting averaging separately for each ball-strike count

Let's create a heatmap that can visualize the batting average for each ball-strike count. To do this, we will use the seaborn `sns.heatmap` function on the `ba_df` we just created. 



## 1.3 Converting data from wide to long format

Another useful operation in pandas is to pivot data from wide format to long format. This can be useful for a number of reason including that it makes it easier to visualize the data using seaborn or plotly. It can also be useful in some cases where we want to join DataFrames together (e.g., on lab 4). 

To pivot the `ba_df` DataFrame from wide format to long format, we will use the `melt` method which takes the following arguments: 

1. `id_vars`: The columns to use as identifier variables. In this case, we will use the `strikes` column.
2. `var_name`: The name to use for the variable column. In this case, we will use `balls`.
3. `value_name`: The name to use for the value column. In this case, we will use `batting_average`. 

Let's use the `melt` method on the `ba_df` DataFrame so that we have columns for the number of strikes, the number of balls, and the values are the batting average. To do this, we will first reset the index of the `ba_df` DataFrame so that the `strikes` column is a regular column and not an index. Then we will use the `melt` method to pivot the DataFrame from wide format to long format. 


In [10]:

# reset the index to create the ba_df2 DataFrame




# Create a long format version of the data




## 1.4 Pivoting back from long to wide format

We can also pivot the data back from long format to wide format using the `pivot()` method. The `pivot` method takes the following arguments:

1. `index`: The column to use as the index. In this case, we will use the `strikes` column.
2. `columns`: The column to use as the columns. In this case, we will use the `balls` column.
3. `values`: The column to use as the values. In this case, we will use the `batting_average` column.

Let's pivot the `long_df` DataFrame back to wide format using the `pivot()` method. This will create a DataFrame where the index is the number of strikes, the columns are the number of balls, and the values are the batting average. 

# Part 2: Interactive graphics

In lab 3 you used percentiles and data visualizations to assess which batting statistics values were impressive for a particular statistic. While the function you wrote to get percentiles from a DataFrame was useful, it is often useful to be able to interactively explore the data to find impressive statistics. Let's continue exploring impressive statistics but let's use interactive graphics to see how this could make it easier to see which statistics are most impressive. 

Below we load the Lahman Batting data which is the data we will use for these exercises. 



In [11]:
batting = pd.read_csv("Batting.csv")
players = pd.read_csv("People.csv")
batting = batting.merge(players)

batting.head()


,playerID,yearID,stint,teamID,lgID,G,G_batting,AB,R,H,...,nameLast,nameGiven,weight,height,bats,throws,debut,bbrefID,finalGame,retroID
0,aardsda01,2004,1,SFN,NL,11,NaN,0,0,0,...,Aardsma,David Allan,215.0,75.0,R,R,2004-04-06,aardsda01,2015-08-23,aardd001
1,aardsda01,2006,1,CHN,NL,45,NaN,2,0,0,...,Aardsma,David Allan,215.0,75.0,R,R,2004-04-06,aardsda01,2015-08-23,aardd001
2,aardsda01,2007,1,CHA,AL,25,NaN,0,0,0,...,Aardsma,David Allan,215.0,75.0,R,R,2004-04-06,aardsda01,2015-08-23,aardd001
3,aardsda01,2008,1,BOS,AL,47,NaN,1,0,0,...,Aardsma,David Allan,215.0,75.0,R,R,2004-04-06,aardsda01,2015-08-23,aardd001
4,aardsda01,2009,1,SEA,AL,73,NaN,0,0,0,...,Aardsma,David Allan,215.0,75.0,R,R,2004-04-06,aardsda01,2015-08-23,aardd001


## 2.1 Plotting batting statistics

Let's create a function that will plot a histogram of a particular batting statistic and show the percentile value for that statistic. The function will take the following arguments:

1. `statistic`: The batting statistic to plot (e.g., "HR", "H", "2B", "3B", "SB").
2. `percentile`: The percentile to show on the plot (e.g., 90).
3. `min_AB`: The minimum number of at-bats to consider a player for the plot (e.g., 300).

The function will create a histogram of the specified batting statistic, show the percentile value as a vertical line on the plot, and return the percentile value. We will then use `ipywidgets` to create an interactive widget that allows us to change the statistic, percentile, and minimum number of at-bats through a graphical interface. 

In [12]:
def plot_stats(statistic = "HR", percentile = 90, min_AB = 300):

    ...
    
    # Create a filtered version of the batting data to a particular year range and number of at-bats
    

    # Get the percentile value


    # Visualize the data as a histogram 


    

plot_stats(statistic = "HR", percentile = 90, min_AB = 300)


## 2.2 Interactive widgets

Now that we have the `plot_stats` function, we can use `ipywidgets` to create an interactive widget that allows us to change the statistic, percentile, and minimum number of at-bats through a graphical interface using the `widgets.interact` function. 


## 2.3 Interactive widgets

A slightly more complex version of the `plot_stats` function is below. This version allows you to specify a year range and whether to show a boxplot or histogram. It also rounds the percentile value to an integer if it is an integer, otherwise it rounds to 3 decimal places. 

If you find this interesting, please explore this further on your own where you could add additional statistics (e.g., OBP, SLG, OPS), look at pitching statistics, etc.


In [13]:
def plot_stats2(statistic = "HR", percentile = 90, min_AB = 300, year_range = (1871, 2024), show_boxplot = False):
    
    # Create a filtered version of the batting data to a particular year range and number of at-bats
    batting2 = batting.copy()
    batting2 = batting2[batting2.AB >= min_AB]
    batting2 = batting2[batting2.yearID >= year_range[0]]
    batting2 = batting2[batting2.yearID <= year_range[1]]
    
    percentile_value = batting2[statistic].quantile(percentile/100)

    # if percentile_value is an integer round it to not show decimal places
    if (round(percentile_value) == percentile_value):
        percentile_value = int(percentile_value)
    else:
        percentile_value = round(percentile_value, 3)

    #
    if show_boxplot:
        plt.boxplot(batting[statistic], vert=False)
    else:
        plt.hist(batting[statistic], edgecolor = "k", bins = 20, color="blue", alpha=0.5)

    plt.title(f"{statistic} {percentile}th percentile is: {percentile_value}")
    plt.xlabel(f"{statistic}")
    plt.ylabel("Count")
    plt.axvline(percentile_value, color='red', linestyle='dashed', linewidth=1)
    
    return percentile_value
    

import ipywidgets as widgets
from IPython.display import display

widgets.interact(
    plot_stats2,
    statistic=["G", "HR", "H", "2B", "3B", "SB"],
    percentile=(0, 100),
    min_AB = 300,
    year_range=widgets.IntRangeSlider(
        value=[1871, 2024],
        min=1871,
        max=2024,
        step=1,
        description="Years",
        continuous_update=False
    )
);


interactive(children=(Dropdown(description='statistic', index=1, options=('G', 'HR', 'H', '2B', '3B', 'SB'), v…

## 2.4 Interactive graphics with plotly

We can also create interactive graphics using the `plotly` package. This allows us to create scatter plots, line plots, and other types of plots that can be interacted with in a web browser where we can hover over points to see more information, zoom in and out, and save the plot as an image. 

Let's create a scatter plot of the maximum home runs by season, where it will show the player's first and last name of the player who hit the maximum number of home runs in that season when we hover over the point. 

To start, let's create a DataFrame called `max_players` that contains the player who hit the maximum number of home runs in each season. We will do this by grouping the `batting` DataFrame by `yearID` and then using the `idxmax` method to get the index of the row with the maximum number of home runs for each year. We will then use this index to select the rows from the `batting` DataFrame that correspond to the player who had the maximum number of home runs in that season. 



## 2.5 Create an interactive scatter plot of the maximum home runs by season

Now that we have the `max_players` DataFrame, we can create an interactive scatter plot using `plotly.express`. The scatter plot will have the x-axis as the year, the y-axis as the maximum number of home runs, and when we hover over a point, it will show the player's first and last name along with the year. 


In [14]:
import plotly.express as px










In [15]:
%%capture

# Note: Jupyter notebooks with interactive graphics do not render well as pdfs
!quarto render class_05.ipynb --cache-refresh --to pdf 
